### IMPORT THE LIBRARIES AND TOOLS REQUIRED

In [38]:
import os
from dotenv import load_dotenv

#data ingestion libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#embeddings

from langchain_community.embeddings import JinaEmbeddings

In [39]:
load_dotenv()  # Load environment variables from .env file

True

## fetch the api keys from .env file

In [40]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

## loading the data


In [41]:
data_path = os.path.join("data","college-faq.txt")

# data ingestion

In [42]:
loader = TextLoader(data_path, encoding="utf-8", autodetect_encoding=True)

docs = loader.load()

print(f"Loaded {len(docs)} documents from {data_path}")

# Print the first 500 characters of the first document

# print(docs[0].page_content[:500])  

Loaded 1 documents from data\college-faq.txt


# splitting the data

In [43]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks.")

Split into 2 chunks.


# chunks and its data

In [44]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content[:100]}...")  # Print the first 100 characters of each chunk

Chunk 1: ==============================
COLLEGE ADMISSIONS

Applications open ...
Chunk 2: Late fee: 2 INR per book per day.


COLLEGE EXAMINATIONS
============...


## embeddings

In [45]:
vectors = JinaEmbeddings(jina_key=jina_key, model_name="jina-embedding-v2-base-en")

print("Generating embeddings for chunks...", vectors.model_name)

Generating embeddings for chunks... jina-embedding-v2-base-en


# store data in vector db

In [46]:
from langchain_community.vectorstores import FAISS  # facebook AI Similarity Search

In [47]:
 # from documnents asks for a list of documents and embeddings asks for a list of embeddings. 
 # The FAISS vector store will create an index of the embeddings and allow for efficient similarity search.

vectors = JinaEmbeddings(
   jina_api_key=jina_key,
   model_name="jina-embeddings-v2-base-en"
)

vector_store = FAISS.from_documents(chunks, vectors)

print("Vector store created with FAISS.",vector_store.index.ntotal)

Vector store created with FAISS. 2


## It will do a similarity search for every chunk

In [49]:
query = "What is the admission process for international students?"

top_match = vector_store.similarity_search(query, k=2)

print(f"Top {len(top_match)} matches for query: '{query}'")
for i, match in enumerate(top_match):
    print(f"Match {i+1}: {match.page_content}...")  # Print the first 200 characters of each match

    



Top 2 matches for query: 'What is the admission process for international students?'
Match 1: ==============================
COLLEGE ADMISSIONS

Applications open June 1 and close July 31.

Required documents:
- Application form
- Academic transcripts
- Government ID
- Passport-size photograph

Application fee: 500 INR.


COLLEGE LIBRARY

Library hours:
Monday to Saturday
8:00 AM - 8:00 PM

Students can borrow up to three books.

Borrowing period: 14 days.

Late fee: 2 INR per book per day....
Match 2: Late fee: 2 INR per book per day.


COLLEGE EXAMINATIONS

Students should arrive 30 minutes before an examination.

College ID is required.

Phones and smart watches are not permitted....


## data retrieval pipeline

In [50]:
from langchain_groq import ChatGroq

In [57]:
llm = ChatGroq(
   model = "openai/gpt-oss-120b",
   temperature = 0.4 # creativity of a model's responses. Lower values make it more deterministic, while higher values make it more creative.
)

llm.model_name

'openai/gpt-oss-120b'

In [63]:
result = llm.invoke("explain about ai in 20 words")

In [64]:
result.content

'Artificial intelligence replicates human thought, allowing computers to learn, reason, perceive, and act intelligently across many industries like healthcare, finance.'